In [2]:
import sys
sys.path.append(rf"/Users/baia/Desktop/PYTHON/mba_dsa_usp_esalq")

from TCC.utils.constantes import *
import matplotlib.pyplot as plt

## S&P500 Close Price
- Retorno logarítmico diário do índice S&P 500. Representa o desempenho das 500 maiores empresas de capital aberto dos EUA.

- É o Beta do Mercado. Na teoria moderna de portfólio, todo ativo tem uma correlação com o mercado (S&P 500). O objetivo é testar se o Bitcoin migrou de "Ativo Descorrelacionado" (Alpha puro) para "Ativo de Alto Beta" (Tech Stock).

- Fonte: https://br.tradingview.com/symbols/SPX/

### HIPÓTESES
- Hipótese (Era Varejo): Desacoplamento (Uncorrelated). Pré-2020, o Bitcoin era movido por fatores endógenos (Halving, Tether, Hacks). Uma queda de 2% no S&P 500 raramente derrubava o Bitcoin no mesmo dia. Espera-se correlação próxima de zero.

- Hipótese (Era Institucional): Acoplamento Direcional (Risk-On/Off). Pós-2020, algoritmos de paridade de risco vendem tudo quando o S&P cai. 1. S&P 500 Cai: Bitcoin cai junto. Se cair muito, investidores vendem Altcoins para cobrir margem ou fugir do risco -> BTC.D Sobe (ou cai menos que Altcoins).2. S&P 500 Sobe: Apetite por risco volta. Dinheiro flui para as pontas mais arriscadas (Crypto) -> BTC.D Cai (Altseason).

### TRATAMENTO
- Antes de calcular qualquer variação, precisamos preencher esses buracos (valores NaNs aos finais de semana). A técnica padrão é o Forward Fill (ffill): assume-se que o preço de sábado e domingo é igual ao fechamento de sexta-feira. Sem isso, você perderá 30% dos dados da sua série temporal ao alinhar com o Bitcoin.
- Preço com tendência de alta (Não estacionário). Exige Log-Retorno.

In [21]:
df_sp500 = (pd.read_csv(rf"raw/2014_SP500_PRICE.csv")

                         .assign(Data_UTC = lambda df: pd.to_datetime(df['time'], utc=True))
                         .assign(Data_UTC = lambda df: df['Data_UTC'].dt.strftime("%Y-%m-%d"))

                        .rename(columns={'close': 'close_price'}) 

                        [['Data_UTC', 'close_price']] 
 
)

df_sp500_log_Ret =(
    df_periodo
        .merge(df_sp500, how='left', on='Data_UTC')
        # Preço se mantém ao FDS
        .assign(close_price = lambda df: df['close_price'].fillna(method='ffill'))
        .assign(SPX_Log_Ret = lambda df: np.log(df['close_price']) - np.log(df['close_price'].shift(1)))


        .query("Data_UTC > '2017-01-03'")
        [['Data_UTC','close_price','SPX_Log_Ret']]

)
df_sp500_log_Ret

# print_dataframe_info(df_sp500_log_Ret, "S&P 500 Price")

/var/folders/h1/_3z4z04x4j17zlfj224w53hr0000gn/T/ipykernel_6114/3592607401.py:16: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .assign(close_price = lambda df: df['close_price'].fillna(method='ffill'))


,Data_UTC,close_price,SPX_Log_Ret
4,2017-01-04,2270.75,0.005706
5,2017-01-05,2269.00,-0.000771
6,2017-01-06,2276.98,0.003511
7,2017-01-07,2276.98,0.000000
8,2017-01-08,2276.98,0.000000
...,...,...,...
3131,2025-07-28,6389.76,0.000174
3132,2025-07-29,6370.87,-0.002961
3133,2025-07-30,6362.89,-0.001253
3134,2025-07-31,6339.38,-0.003702


## DXY
- Retorno diário do Índice Dólar (DXY). O índice mede a força do USD contra uma cesta de moedas fortes (Euro, Iene, Libra, etc.)

- O Dólar é o ativo de "refúgio supremo". Quando o DXY sobe, significa que o capital global está fugindo de risco e indo para caixa (Cash is King), o que geralmente drena liquidez de ações e cripto. Quando o DXY cai, a liquidez global se expande, favorecendo ativos de risco.

- Fonte: https://br.tradingview.com/symbols/TVC-DXY/

### HIPÓTESES
- Hipótese (Era Varejo)Desacoplamento (Baixa Causalidade). Pré-2020, o Bitcoin vivia em seu próprio mundo. Movimentos no Dólar não afetavam o "Hype" das ICOs ou halving. Espera-se que o DXY tenha baixa importância (SHAP value baixo) ou correlação aleatória.

- Hipótese (Era Institucional)Acoplamento Negativo Forte (Risk-Off). Pós-2020, com a entrada de fundos macro, o BTC passou a ser negociado como "Tech Stock". Expectativa: DXY Sobe -> BTC.D sobe e Preço caem (Institucional vende risco, ou seja altcoins). DXY Cai -> BTC.D cai e Preço sobem (Institucional toma risco).

### TRATAMENTO
- O tratamento deve ser robusto. O Forward Fill (ffill) é obrigatório para garantir que, se o dado de segunda faltar, usamos o de domingo/sexta. O mercado cripto não pode ter "buracos".

- Transformação: Como é um índice de preço (Base 100), usamos Log-Retorno.

In [35]:
df_dxy = (pd.read_csv(rf"raw/201501_DXY.csv")

                        .assign(Data_UTC = lambda df: pd.to_datetime(df['time'],unit='s', utc=True))
                        .assign(Data_UTC = lambda df: df['Data_UTC'].dt.strftime("%Y-%m-%d"))
                        .rename(columns={'close': 'close_price'}) 

                        [['Data_UTC', 'close_price']] 
 
)

df_dxy_log_ret =(
    df_periodo
        .merge(df_dxy, how='left', on='Data_UTC')
        .assign(close_price = lambda df: df['close_price'].fillna(method='ffill'))
        .assign(DXY_Log_Ret = lambda df: np.log(df['close_price']) - np.log(df['close_price'].shift(1)))

        .query("Data_UTC > '2017-01-03'")
        [['Data_UTC','close_price','DXY_Log_Ret']]

)
df_dxy_log_ret
# print_dataframe_info(df_dxy_log_ret, "S&P 500 Price")

/var/folders/h1/_3z4z04x4j17zlfj224w53hr0000gn/T/ipykernel_6114/3694519528.py:15: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .assign(close_price = lambda df: df['close_price'].fillna(method='ffill'))


,Data_UTC,close_price,DXY_Log_Ret
4,2017-01-04,102.700,-0.004954
5,2017-01-05,101.520,-0.011556
6,2017-01-06,102.220,0.006872
7,2017-01-07,102.220,0.000000
8,2017-01-08,101.930,-0.002841
...,...,...,...
3131,2025-07-28,98.898,0.002369
3132,2025-07-29,99.887,0.009951
3133,2025-07-30,100.050,0.001631
3134,2025-07-31,98.684,-0.013747


## NASDAQ
- Retorno diário do índice NASDAQ-100 (as 100 maiores empresas não-financeiras da bolsa americana).
- Serve como proxy para o "Sentimento do Setor de Tecnologia". Como o Bitcoin e o Ethereum são tecnologias emergentes, eles competem pelo mesmo tipo de capital de risco que a Apple, Nvidia e Microsoft.

- Fonte: https://br.tradingview.com/symbols/NASDAQ-NDX/

### HIPÓTESES
- Hipótese (Era Varejo) Correlação Baixa/Nula. Pré-2020, o Bitcoin era um ativo "exótico". Um crash na Apple ou Google não causava venda automática de Bitcoin, pois os investidores eram grupos diferentes (Cypherpunks/Varejo vs. Wall Street).

- Hipótese (Era Institucional) Alto Acoplamento (High Correlation). Pós-2020, algoritmos de trading operam cestas de ativos. Cenário de Alta: NASDAQ sobe -> Apetite por risco aumenta -> Capital flui para Crypto (Altcoins sobem mais que BTC) -> BTC.D Cai. Cenário de Baixa: NASDAQ cai -> Aversão a risco -> Liquidação geral (Crypto cai, mas Altcoins sangram mais) -> BTC.D Sobe (relativamente).

### TRATAMENTO
- Aplica-se a imputação de dias faltantes (ffill) para garantir o alinhamento temporal. O log-retorno mede a variação percentual contínua, permitindo testar a hipótese de acoplamento do Bitcoin como um ativo de tecnologia ("High Beta Tech Stock").

In [42]:
df_nasdaq = (pd.read_csv(rf"raw/201501_NASDAQ.csv")

                        .assign(Data_UTC = lambda df: pd.to_datetime(df['time'],unit='s', utc=True))
                        .assign(Data_UTC = lambda df: df['Data_UTC'].dt.strftime("%Y-%m-%d"))
                        .rename(columns={'close': 'close_price'}) 

                        [['Data_UTC', 'close_price']] 
 
)
df_nasdaq

df_nasdaq_log_ret =(
    df_periodo
        .merge(df_nasdaq, how='left', on='Data_UTC')
        .assign(close_price = lambda df: df['close_price'].ffill())
        .assign(NASDAQ_Log_Ret = lambda df: np.log(df['close_price']) - np.log(df['close_price'].shift(1)))

        .query("Data_UTC > '2017-01-03'")
        [['Data_UTC','close_price','NASDAQ_Log_Ret']]

)
df_nasdaq_log_ret
# print_dataframe_info(df_nasdaq_log_ret, "NASDAQ Price")

,Data_UTC,close_price,NASDAQ_Log_Ret
4,2017-01-04,4934.4,0.006200
5,2017-01-05,4963.6,0.005900
6,2017-01-06,5008.1,0.008925
7,2017-01-07,5008.1,0.000000
8,2017-01-08,5013.5,0.001078
...,...,...,...
3393,2025-07-28,23332.7,-0.002303
3394,2025-07-29,23593.0,0.011094
3395,2025-07-30,23158.1,-0.018605
3396,2025-07-31,22746.7,-0.017925


## GOLD
- Retorno diário do preço do Ouro à vista (Spot) ou Futuros. O Ouro é o ativo de segurança clássico contra incerteza geopolítica e inflação monetária.
- Testa a narrativa do "Ouro Digital". Em momentos de crise (inflação alta ou guerra), o Ouro sobe. A questão é: o Bitcoin sobe junto (comportamento institucional de proteção) ou cai junto com ações de tecnologia (comportamento de risco)?

- Fonte: https://br.tradingview.com/symbols/XAUUSD/

### HIPÓTESES
- Hipótese (Era Varejo) Descorrelação (Indiferença). Pré-2020, o Bitcoin era "dinheiro mágico da internet". Investidores de Ouro não compravam BTC e vice-versa. Movimentos no Ouro tinham pouco ou nenhum impacto estatístico na Dominância ou preço do BTC.

- Hipótese (Era Institucional) Correlação Positiva Condicional. Pós-2020 (Narrativa Paul Tudor Jones/BlackRock):1. Cenário de Inflação: Ouro Sobe -> BTC Sobe -> BTC.D Sobe (Investidores buscam escassez, fugindo de Altcoins inflacionárias).2. Cenário de Pânico (Crash): Se o Ouro sobe por medo extremo (guerra), o Bitcoin ainda pode cair inicialmente (como ativo de risco), mas tende a segurar valor melhor que Altcoins, mantendo a Dominância estável ou alta.

### TRATAMENTO
- Log-Retorno (após Forward Fill)
- Justificativa: Como ativo tradicional, exige imputação de valores (ffill) para fins de semana/feriados. O log-retorno permite a comparação direta da volatilidade e correlação com o Bitcoin, testando a hipótese do BTC como "Ouro Digital".

In [45]:
df_gold = (pd.read_csv(rf"raw/201501_PRICE_GOLD.csv")

                        .assign(Data_UTC = lambda df: pd.to_datetime(df['time'],unit='s', utc=True))
                        .assign(Data_UTC = lambda df: df['Data_UTC'].dt.strftime("%Y-%m-%d"))
                        .rename(columns={'close': 'close_price'}) 

                        [['Data_UTC', 'close_price']] 
 
)
df_gold

df_gold_log_ret =(
    df_periodo
        .merge(df_gold, how='left', on='Data_UTC')
        .assign(close_price = lambda df: df['close_price'].ffill())
        .assign(GOLD_Log_Ret = lambda df: np.log(df['close_price']) - np.log(df['close_price'].shift(1)))

        .query("Data_UTC > '2017-01-03'")
        [['Data_UTC','close_price','GOLD_Log_Ret']]

)
df_gold_log_ret
# print_dataframe_info(df_gold_log_ret, "GOLD Price")

,Data_UTC,close_price,GOLD_Log_Ret
4,2017-01-04,1180.200,0.014337
5,2017-01-05,1172.130,-0.006861
6,2017-01-06,1172.130,0.000000
7,2017-01-07,1172.130,0.000000
8,2017-01-08,1180.750,0.007327
...,...,...,...
3131,2025-07-28,3327.115,0.004041
3132,2025-07-29,3275.817,-0.015538
3133,2025-07-30,3289.830,0.004269
3134,2025-07-31,3362.640,0.021890


## US10Y
- Diferença diária da taxa de rendimento dos títulos do Tesouro Americano de 10 anos. Não usamos retornos logarítmicos para taxas de juros, usamos a diferença aritmética (pontos base).

- É o denominador do Valuation global. Todo ativo de risco (Ações, Imóveis, Bitcoin) compete com essa taxa. Se o governo americano paga 5% sem risco, por que arriscar em Altcoins?

- Fonte: https://br.tradingview.com/symbols/TVC-US10Y/

### HIPÓTESES
- Hipótese (Era Varejo): Baixa Sensibilidade. Pré-2020, o Bitcoin era um ativo de nicho e o ambiente macro era de juros baixos/estáveis. O trader de varejo ignorava o mercado de títulos (Bond Market).

- Hipótese (Era Institucional): Alta Causalidade (Risk-Off). Pós-2020, instituições usam modelos de Fluxo de Caixa Descontado (DCF).
    - 1. US10Y Dispara: O valor presente dos ativos de risco cai. O mercado vende Crypto. Altcoins sofrem mais (BETA maior), então o Bitcoin pode segurar valor melhor (BTC.D Sobe).
    - 2. US10Y Cai: O dinheiro fica barato. Busca por rendimento (Yield Hunting). Capital flui para Altcoins/DeFi (BTC.D Cai).

### TRATAMENTO
- Primeira Diferença (após Forward Fill)
- Justificativa: Tratando-se de uma taxa percentual, utiliza-se a diferença simples para capturar a variação em pontos base (basis points). A imputação (ffill) corrige os dias sem negociação (fins de semana) para alinhar com o mercado cripto.

In [50]:
df_us10y = (pd.read_csv(rf"raw/201501_US10Y.csv")

                        .assign(Data_UTC = lambda df: pd.to_datetime(df['time'],unit='s', utc=True))
                        .assign(Data_UTC = lambda df: df['Data_UTC'].dt.strftime("%Y-%m-%d"))
                        .rename(columns={'close': 'close_price'}) 

                        [['Data_UTC', 'close_price']] 
 
)
df_us10y

df_us10y_diff =(
    df_periodo
        .merge(df_us10y, how='left', on='Data_UTC')
        .assign(close_price = lambda df: df['close_price'].ffill())
        .assign(US10Y_Diff = lambda df: df['close_price'].diff())

        .query("Data_UTC > '2017-01-03'")
        [['Data_UTC','close_price','US10Y_Diff']]

)
df_us10y_diff
# print_dataframe_info(df_us10y_diff, "US10Y Price")

,Data_UTC,close_price,US10Y_Diff
4,2017-01-04,2.4413,-0.0059
5,2017-01-05,2.3479,-0.0934
6,2017-01-06,2.4221,0.0742
7,2017-01-07,2.4221,0.0000
8,2017-01-08,2.4221,0.0000
...,...,...,...
3131,2025-07-28,4.3240,-0.0900
3132,2025-07-29,4.3720,0.0480
3133,2025-07-30,4.3820,0.0100
3134,2025-07-31,4.2160,-0.1660


## VIX
- Retorno logarítmico diário do índice VIX. O VIX não é um ativo investível diretamente (embora existam futuros), é uma estatística.
- Mede o custo do seguro de carteira (Puts). Quando o VIX explode, significa que grandes fundos estão pagando qualquer preço para proteger suas carteiras. É o sinal mais puro de Pânico.

- Fonte: https://br.tradingview.com/symbols/TVC-VIX/

### HIPÓTESES
- Hipótese (Era Varejo): Desacoplamento. Pré-2020, o pânico em Wall Street (ex: Guerra Comercial EUA-China) nem sempre chegava em Cripto com a mesma intensidade. O Bitcoin tinha seus próprios ciclos de medo (ex: Banimento na China, Hacks) que não apareciam no VIX.

- Hipótese (Era Institucional): Correlação Positiva com Dominância. Pós-2020, a tese é de "Fuga para Qualidade Relativa".
    - 1. VIX Dispara (Pânico): Liquidação geral. Altcoins (ilíquidas e arriscadas) são vendidas primeiro e mais forte. O Bitcoin cai, mas cai menos ("Ouro Digital"). Resultado: BTC.D Sobe.
    - 2. VIX Cai (Complacência): O mercado relaxa. Busca por Yield e Alpha. Dinheiro flui para Altcoins. Resultado: BTC.D Cai.

### TRATAMENTO
- O VIX mede volatilidade implícita (já é uma porcentagem). No entanto, diferentemente da taxa de juros (US10Y) que se move em pontos base lentos, o VIX tem explosões exponenciais.
- Exemplo: Ir de 12 para 24 (+100%) é um choque de pânico maciço. Ir de 60 para 72 (+20%) é grave, mas o mercado já está em pânico.
- O Log-Retorno é superior aqui. Ele captura a "Aceleração do Medo" em termos percentuais, normalizando os picos de 2020 (Covid) com os picos menores de 2018.
- Tratamento: ffill (obrigatório para fins de semana) + Log-Return

In [54]:
df_vix = (pd.read_csv(rf"raw/201501_VIX.csv")

                        .assign(Data_UTC = lambda df: pd.to_datetime(df['time'],unit='s', utc=True))
                        .assign(Data_UTC = lambda df: df['Data_UTC'].dt.strftime("%Y-%m-%d"))
                        .rename(columns={'close': 'close_price'}) 

                        [['Data_UTC', 'close_price']] 
 
)
df_vix

df_vix_log_ret =(
    df_periodo
        .merge(df_vix, how='left', on='Data_UTC')
        .assign(close_price = lambda df: df['close_price'].ffill())
        .assign(VIX_Log_Ret = lambda df: np.log(df['close_price']) - np.log(df['close_price'].shift(1)))

        .query("Data_UTC > '2017-01-03'")
        [['Data_UTC','close_price','VIX_Log_Ret']]

)
df_vix_log_ret
# print_dataframe_info(df_vix_log_ret, "VIX Price")

,Data_UTC,close_price,VIX_Log_Ret
4,2017-01-04,11.85,-0.081016
5,2017-01-05,11.67,-0.015306
6,2017-01-06,11.32,-0.030450
7,2017-01-07,11.32,0.000000
8,2017-01-08,11.32,0.000000
...,...,...,...
3131,2025-07-28,15.04,0.008011
3132,2025-07-29,15.97,0.059999
3133,2025-07-30,15.47,-0.031809
3134,2025-07-31,16.71,0.077105
